In [1]:
%pip install fiftyone==0.24.1

In [2]:
import fiftyone as fo
import glob, json
from PIL import Image

# Training Dataset

In [3]:
images = glob.glob(r"C:\Users\ashar\repos\ai4doc\dataset\train\images\*.png")
images = sorted(images)

anno = glob.glob(r"C:\Users\ashar\repos\ai4doc\dataset\train\annotations\*.json")
anno = sorted(anno)

In [4]:
def create_samples(images, anno):
    # Create samples for your data
    samples = []
    for im, an in zip(images, anno):
        # Open an image file
        with Image.open(im) as img:
            im_width, im_height = img.size

        # Open the file and load its content as a Python dictionary
        with open(an, 'r') as file:
            an_json = json.load(file)
        
        sample = fo.Sample(filepath=im)

        # Convert detections to FiftyOne format
        detections = []
        for obj in an_json:
            index = obj["id"]

            # Bounding box coordinates should be relative values
            # in [0, 1] in the following format:
            # [top-left-x, top-left-y, width, height]

            # The box is currently of the format x0, y0, x1, y1
            zoom_factor = 10
            width = obj['box'][2] - obj['box'][0]
            height = obj['box'][3] - obj['box'][1]
            bounding_box = [
                zoom_factor*obj["box"][0]/im_width, 
                zoom_factor*obj["box"][1]/im_height, 
                zoom_factor*width/im_width, 
                zoom_factor*height/im_height
                ]

            detections.append(
                fo.Detection(index=index, bounding_box=bounding_box)
            )

        # Store detections in a field name of your choice
        sample["ground_truth"] = fo.Detections(detections=detections)

        samples.append(sample)

    return samples

In [5]:
# Create dataset
samples = create_samples(images, anno)
dataset = fo.Dataset("ai4doc-train-dataset", overwrite=False)
dataset.add_samples(samples)

 100% |█████████████████| 238/238 [7.3s elapsed, 0s remaining, 55.5 samples/s]       


['66b5780f6b040c50d248e067',
 '66b5780f6b040c50d248e068',
 '66b5780f6b040c50d248e069',
 '66b5780f6b040c50d248e06a',
 '66b5780f6b040c50d248e06b',
 '66b5780f6b040c50d248e06c',
 '66b5780f6b040c50d248e06d',
 '66b5780f6b040c50d248e06e',
 '66b5780f6b040c50d248e06f',
 '66b5780f6b040c50d248e070',
 '66b5780f6b040c50d248e071',
 '66b5780f6b040c50d248e072',
 '66b5780f6b040c50d248e073',
 '66b5780f6b040c50d248e074',
 '66b5780f6b040c50d248e075',
 '66b5780f6b040c50d248e076',
 '66b5780f6b040c50d248e077',
 '66b578106b040c50d248e078',
 '66b578106b040c50d248e079',
 '66b578106b040c50d248e07a',
 '66b578106b040c50d248e07b',
 '66b578106b040c50d248e07c',
 '66b578106b040c50d248e07d',
 '66b578106b040c50d248e07e',
 '66b578106b040c50d248e07f',
 '66b578106b040c50d248e080',
 '66b578106b040c50d248e081',
 '66b578106b040c50d248e082',
 '66b578106b040c50d248e083',
 '66b578116b040c50d248e084',
 '66b578116b040c50d248e085',
 '66b578116b040c50d248e086',
 '66b578116b040c50d248e087',
 '66b578116b040c50d248e088',
 '66b578116b04

In [7]:
session = fo.launch_app(dataset, port=5151)

In [7]:
session.close()

In [13]:
# Step 3: Send samples to CVAT

# A unique identifier for this run
anno_key = "cvat_basic_recipe"

dataset.annotate(
    anno_key,
    label_field="ground_truth",
    launch_editor=True,
)
print(dataset.get_annotation_info(anno_key))

Please enter your login credentials.
You can avoid this in the future by setting your `FIFTYONE_CVAT_USERNAME` and `FIFTYONE_CVAT_PASSWORD` environment variables


KeyError: 'classes'

In [ ]:
# Step 5: Merge annotations back into FiftyOne dataset

dataset = fo.load_dataset("cvat-annotation-example")
dataset.load_annotations(anno_key)

# Load the view that was annotated in the App
view = dataset.load_annotation_view(anno_key)
session = fo.launch_app(view=view)

In [12]:
# Step 6: Cleanup

# Delete tasks from CVAT
results = dataset.load_annotation_results(anno_key)
results.cleanup()

# Delete run record (not the labels) from FiftyOne
dataset.delete_annotation_run(anno_key)

Deleting tasks...
 100% |█████████████████████| 1/1 [2.0s elapsed, 0s remaining, 0.5 samples/s] 
